# Pipeline Walkthrough: How the Matching Engine Works

**Thesis:** *Design and Evaluation of an AI-Based Talent Matching System for Vocational Apprenticeships* — Samuel Hanok Anchan, SRH University Hamburg.

This notebook runs the complete matching pipeline, one stage at a time, on a small set of **synthetic** sample data (10 students, 8 vacancies). It is meant to show *how the system works*, not to reproduce the thesis results. Every cell executes top to bottom with no API key and no network access.

The engine is a **two-stage retrieval pipeline with a learned re-ranker on top**:

| Stage | What it does | Module |
|---|---|---|
| **1. Hard filters** | SQL-style eligibility gates: city, start date, training track, language (CEFR) | `src/filters.py` |
| **2. Semantic retrieval** | Embed each profile and vacancy, rank by cosine similarity (k-NN) | `src/demo_embedding.py` |
| **3. Learned re-ranker** | LightGBM LambdaRank over the signals the embedding cannot see | `src/rerank.py`, `src/features.py` |

> **On the embedding step.** In production, Stage 2 uses Google `gemini-embedding-001` (768-d) over an API, and the production embedding text depends on the proprietary ESCO occupation catalogue. Neither is shipped here. So this notebook substitutes a small, deterministic, offline embedder (`src/demo_embedding.py`) built only on scikit-learn. **The pipeline shape is identical**; only the embedding model differs. See the README for the full note.

## 0. Setup

In [1]:
import json, sys
from pathlib import Path
import numpy as np
sys.path.insert(0, '..')   # so `import src.*` resolves from the repo root

from src.filters import is_eligible, eligible_jobs, meets_all_language_requirements, \
    start_date_compatible, category_compatible
from src.baseline import TfidfBaseline
from src.features import build_features, FEATURE_NAMES, language_gap_min
from src.rerank import train_reranker, feature_importance
from src import demo_embedding as de

DATA = Path('..') / 'sample_data'
students = json.loads((DATA / 'sample_students.json').read_text())
jobs     = json.loads((DATA / 'sample_jobs.json').read_text())
apps     = json.loads((DATA / 'sample_applications.json').read_text())
S = {s['student_id']: s for s in students}
J = {j['job_id']: j for j in jobs}
print(f'{len(students)} students, {len(jobs)} vacancies, {len(apps)} company decisions')

10 students, 8 vacancies, 16 company decisions


## 1. The data

Each **student** carries stated career interests, languages *with proficiency*, work experience, a home city and an availability date. Each **vacancy** carries a title, required skills, a city, a start date, a training track and language requirements *with CEFR levels*.

Note the candidate below: their interest is written as *"vehicle mechanic"* in English, while the matching vacancy is titled *"Kfz-Mechatroniker"* in German. A keyword matcher sees no shared word. Watch how Stage 2 handles that.

In [2]:
def show_student(s):
    langs = ', '.join(f"{l['language']} ({l['proficiency'].title()})" for l in s['languages'])
    print(f"STUDENT {s['student_id']}  ({s['matching_category']}, {s['city']}, from {s['start_date']})")
    print(f"  interests : {', '.join(s['career_fields'])}")
    print(f"  languages : {langs}")
    print(f"  work      : {[w['job_title'] for w in s['work_experience']] or 'none'}")

def show_job(j):
    langs = ', '.join(f"{r['language']} {r['level']} ({r['kind'].lower()})" for r in j['language_requirements'])
    print(f"VACANCY {j['job_id']}  ({j['matching_category']}, {j['city']}, starts {j['start_date']})")
    print(f"  title     : {j['title']}")
    print(f"  languages : {langs}")

show_student(S['s01']); print(); show_job(J['j01'])

STUDENT s01  (VOCATIONAL, Berlin, from 2026-09)
  interests : vehicle mechanic, Automotive
  languages : German (Intermediate), English (Fluent), Portuguese (Native)
  work      : ['Workshop assistant']

VACANCY j01  (VOCATIONAL, Berlin, starts 2026-09)
  title     : Ausbildung Kfz-Mechatroniker/in
  languages : German B2 (required), English B1 (optional)


## 2. Stage 1 — hard eligibility filters

Before any similarity is computed, a vacancy is excluded unless it clears **four gates**: same city, a start date on or after the student is available, a compatible training track, and the required-language CEFR tier. These are pass/fail. The code lives in `src/filters.py` and is the exact offline replica of the production SQL gates.

In [3]:
s = S['s01']
print(f"Student {s['student_id']}: {s['city']}, German INTERMEDIATE, available {s['start_date']}\n")
print(f"{'vacancy':<8}{'city':<10}{'track':<8}{'start':<9}{'language':<9}{'ELIGIBLE?'}")
for j in jobs:
    city = 'ok' if s['city']==j['city'] else 'FAIL'
    trk  = 'ok' if category_compatible(s,j) else 'FAIL'
    dt   = 'ok' if start_date_compatible(s,j) else 'FAIL'
    lng  = 'ok' if meets_all_language_requirements(s,j) else 'FAIL'
    print(f"{j['job_id']:<8}{city:<10}{trk:<8}{dt:<9}{lng:<9}{'YES' if is_eligible(s,j) else 'no'}")

elig = eligible_jobs(s, jobs)
print(f"\n-> {len(elig)} of {len(jobs)} vacancies survive Stage 1: {[j['job_id'] for j in elig]}")

Student s01: Berlin, German INTERMEDIATE, available 2026-09

vacancy city      track   start    language ELIGIBLE?
j01     ok        ok      ok       ok       YES
j02     FAIL      ok      ok       ok       no
j03     FAIL      FAIL    ok       FAIL     no
j04     ok        ok      ok       ok       YES
j05     FAIL      ok      ok       ok       no
j06     FAIL      ok      ok       FAIL     no
j07     FAIL      ok      ok       ok       no
j08     ok        ok      ok       ok       YES

-> 3 of 8 vacancies survive Stage 1: ['j01', 'j04', 'j08']


**Filter attrition.** A finding of the thesis is that these hard gates exclude the great majority of pairs recruiters actually acted on, chiefly because students apply *across* cities. We can see the same effect in miniature on the sample decisions.

In [4]:
judged = [(a['student_id'], a['job_id']) for a in apps]
gates = {'city':0,'start_date':0,'category':0,'language':0}
excluded = 0
for sid, jid in judged:
    st, jb = S[sid], J[jid]
    if st['city'] != jb['city']: gates['city'] += 1
    if not start_date_compatible(st, jb): gates['start_date'] += 1
    if not category_compatible(st, jb): gates['category'] += 1
    if not meets_all_language_requirements(st, jb): gates['language'] += 1
    if not is_eligible(st, jb): excluded += 1
n = len(judged)
for g, c in gates.items():
    print(f'  {g:<11} rejects {c}/{n} judged pairs ({c/n*100:.0f}%)')
print(f'  ALL GATES   exclude {excluded}/{n} ({excluded/n*100:.0f}%) of pairs a company actually acted on')

  city        rejects 1/16 judged pairs (6%)
  start_date  rejects 0/16 judged pairs (0%)
  category    rejects 1/16 judged pairs (6%)
  language    rejects 6/16 judged pairs (38%)
  ALL GATES   exclude 7/16 (44%) of pairs a company actually acted on


## 3. Stage 2 — semantic retrieval (embedding + k-NN)

Each profile and vacancy is flattened into a single paragraph, embedded into a unit vector, and ranked by cosine similarity. The paragraph is built by `demo_embedding.student_search_text`.

> **The blind spot, on purpose.** Look closely at the search text: it lists language **names** but never **proficiency levels**. "German (Basic)" and "German (Native)" therefore produce nearly identical vectors. Availability dates and city are absent too. This is exactly the limitation of the production embedding (thesis §3.4.2), reproduced here, and it is the reason Stage 3 exists.

In [5]:
print('Search text the embedder actually sees for s01:')
print(' ', de.student_search_text(S['s01']))
print('\nNotice: "German" appears with no proficiency level attached.')

Search text the embedder actually sees for s01:
  A student interested in vehicle mechanic, Automotive with experience in Workshop assistant, active in Robotics club, with languages including German, English, Portuguese.

Notice: "German" appears with no proficiency level attached.


In [6]:
# embed every vacancy once, then rank all vacancies for one student by cosine
job_ids  = [j['job_id'] for j in jobs]
job_vecs = de.embed([de.job_search_text(j) for j in jobs])

def knn(student, k=5):
    v = de.embed([de.student_search_text(student)])[0]
    scored = sorted(((de.cosine(v, job_vecs[i]), job_ids[i]) for i in range(len(jobs))), reverse=True)
    return scored[:k]

print('Top-5 vacancies for s01 (interest: "vehicle mechanic"), by cosine similarity:')
for c, jid in knn(S['s01']):
    print(f'  {jid}  cos={c:.3f}   {J[jid]["title"]}')

Top-5 vacancies for s01 (interest: "vehicle mechanic"), by cosine similarity:
  j06  cos=0.628   Ausbildung Medizinische/r Fachangestellte/r
  j08  cos=0.584   Ausbildung Industriemechaniker/in
  j03  cos=0.582   Duales Studium Betriebswirtschaft
  j01  cos=0.579   Ausbildung Kfz-Mechatroniker/in
  j04  cos=0.574   Ausbildung Elektroniker/in für Betriebstechnik


The German-language vacancy **Kfz-Mechatroniker** (`j01`) is retrieved for a student who wrote *"vehicle mechanic"* in English, even though the two strings share no whole word. That cross-lingual, cross-vocabulary matching is precisely what a keyword baseline cannot do, and it is the thesis's most secure result. (The local demo embedder is weaker than the production Gemini model, so the exact ordering differs, but the capability is the same.)

## 4. Stage 3 — the learned re-ranker

Stage 2 is blind to language proficiency, dates and location. Stage 3 hands those signals to a **LightGBM LambdaRank** model as features and lets it re-order the candidates. The ten features are computed by `src/features.py`; the model is trained by `src/rerank.py`.

The headline feature is `language_gap_min`: how far a student's weakest required language sits above or below what the vacancy asks for, on the CEFR ladder. Stage 2 cannot see it at all.

In [7]:
# show the feature vector Stage 3 builds for one (student, vacancy) pair
s, j = S['s02'], J['j01']   # s02 has BASIC German; j01 requires German B2
vec = de.embed([de.student_search_text(s)])[0]
cos = de.cosine(vec, job_vecs[job_ids.index('j01')])
tf  = TfidfBaseline(jobs).score(s, ['j01'])['j01']
feats = build_features(s, j, cos, tf)
print(f'Feature vector for (s02 BASIC German) x (j01 needs German B2):')
for name, val in zip(FEATURE_NAMES, feats):
    print(f'  {name:<20} {val:+.3f}')
print(f"\nlanguage_gap_min = {language_gap_min(s, j):+.0f}  (negative = the student is short of the requirement)")

Feature vector for (s02 BASIC German) x (j01 needs German B2):
  cosine_similarity    +0.524
  tfidf_similarity     +0.000
  language_gap_min     -2.000
  meets_all_languages  +0.000
  same_city            +1.000
  start_gap_months     +0.000
  category_match       +1.000
  skill_overlap        +0.000
  n_work_experience    +1.000
  n_languages          +3.000

language_gap_min = -2  (negative = the student is short of the requirement)


In [8]:
# derive gold labels from the sample company decisions
gold = {}
for a in apps:
    gold[(a['student_id'], a['job_id'])] = 0 if a['status'] == 'REJECTED_BY_COMPANY' else 1

# build the training matrix over every (student, vacancy) cell
baseline = TfidfBaseline(jobs)
svecs = {s['student_id']: de.embed([de.student_search_text(s)])[0] for s in students}
rows, ys, sids, meta = [], [], [], []
for s in students:
    tf = baseline.score(s, job_ids)
    for i, jid in enumerate(job_ids):
        cos = de.cosine(svecs[s['student_id']], job_vecs[i])
        rows.append(build_features(s, J[jid], cos, tf.get(jid, 0.0)))
        ys.append(1 if gold.get((s['student_id'], jid)) == 1 else 0)
        sids.append(s['student_id']); meta.append((s['student_id'], jid))
X, y = np.array(rows, float), np.array(ys, int)

# LightGBM needs each student's rows contiguous (one 'group' per student)
order = sorted(range(len(sids)), key=lambda i: sids[i])
sizes, cur, n = [], None, 0
for i in order:
    if sids[i] != cur:
        if cur is not None: sizes.append(n)
        cur, n = sids[i], 0
    n += 1
sizes.append(n)

model = train_reranker(X[order], y[order], sizes)
print('re-ranker trained on the sample.\n')
print('What the model leaned on (share of total gain):')
for name, share in feature_importance(model):
    bar = '#' * int(round(share * 30))
    print(f'  {name:<20} {share*100:5.1f}%  {bar}')

re-ranker trained on the sample.

What the model leaned on (share of total gain):
  cosine_similarity     51.6%  ###############
  same_city             21.0%  ######
  language_gap_min      20.3%  ######
  start_gap_months       2.3%  #
  n_languages            1.8%  #
  meets_all_languages    1.6%  
  category_match         1.4%  
  tfidf_similarity       0.0%  
  skill_overlap          0.0%  
  n_work_experience      0.0%  


The re-ranker leans first on the embedding's own cosine score, then on the structured signals the embedding is blind to — `same_city`, `language_gap_min`, `start_gap_months`. That is the same qualitative story the thesis reports on the full dataset (§4.6), reproduced here on ten synthetic students.

In [9]:
# show Stage 2 -> Stage 3 re-ordering for one student
s = S['s02']
pred = {}
tf = baseline.score(s, job_ids)
for i, jid in enumerate(job_ids):
    cos = de.cosine(svecs[s['student_id']], job_vecs[i])
    f = np.array([build_features(s, J[jid], cos, tf.get(jid, 0.0))], float)
    pred[jid] = float(model.predict(f)[0])
stage2 = [jid for _, jid in sorted(((de.cosine(svecs[s['student_id']], job_vecs[i]), job_ids[i]) for i in range(len(jobs))), reverse=True)]
stage3 = sorted(job_ids, key=lambda jid: -pred[jid])
print(f"Student {s['student_id']} (BASIC German):\n")
print(f"{'rank':<6}{'Stage 2 (embedding)':<28}{'Stage 3 (re-ranked)'}")
for r in range(5):
    print(f'  {r+1:<4}{stage2[r]+"  "+J[stage2[r]]["title"][:20]:<28}{stage3[r]+"  "+J[stage3[r]]["title"][:20]}')

Student s02 (BASIC German):

rank  Stage 2 (embedding)         Stage 3 (re-ranked)
  1   j06  Ausbildung Medizinis   j01  Ausbildung Kfz-Mecha
  2   j03  Duales Studium Betri   j08  Ausbildung Industrie
  3   j04  Ausbildung Elektroni   j03  Duales Studium Betri
  4   j02  Ausbildung Fachinfor   j04  Ausbildung Elektroni
  5   j08  Ausbildung Industrie   j06  Ausbildung Medizinis


## 5. Summary

The three stages ran end to end on synthetic data with no API key:

1. **Hard filters** reduced the candidate set and, as in the thesis, excluded most cross-city pairs.
2. **Semantic retrieval** matched an English *"vehicle mechanic"* to a German *Kfz-Mechatroniker* vacancy.
3. **The learned re-ranker** recovered the signals the embedding discards — language readiness, timing, location.

For the actual experimental results and the figures behind them, see [`01_results_and_figures.ipynb`](01_results_and_figures.ipynb). For the full method, see the thesis.